# ELMo example (Word Embeddings, Part 2, section 3.3.3)

The article's ELMo code uses **AllenNLP**, which was archived in 2022 and only installs with older packages (Python 3.9, see `requirements.txt` in this folder). It is kept in its own notebook so the main notebook can use current versions.

**Added for this repository:** downloads ELMo's published option and weight files. This uses the small ELMo model (54 MB); the original model is `elmo_2x4096_512_2048cnn_2xhighway` (374 MB) from the same address. They are saved under the file names the article uses.

In [1]:
import os, urllib.request
base = 'https://allennlp.s3.amazonaws.com/models/elmo/2x1024_128_2048cnn_1xhighway/'
for remote, local in [('elmo_2x1024_128_2048cnn_1xhighway_options.json', 'elmo_options.json'),
                      ('elmo_2x1024_128_2048cnn_1xhighway_weights.hdf5', 'elmo_weights.hdf5')]:
    if not os.path.exists(local):
        urllib.request.urlretrieve(base + remote, local)
print({f: os.path.getsize(f) for f in ('elmo_options.json', 'elmo_weights.hdf5')})

{'elmo_options.json': 336, 'elmo_weights.hdf5': 54402456}


The article's code:

In [2]:
from allennlp.modules.elmo import Elmo, batch_to_ids

# Initialize ELMo
options_file = 'elmo_options.json'
weight_file = 'elmo_weights.hdf5'
elmo = Elmo(options_file, weight_file, num_output_representations=1)

# Example sentences
sentences = [['I', 'have', 'a', 'green', 'apple'], ['I', 'have', 'a', 'green', 'thumb']]

# Convert sentences to character ids
character_ids = batch_to_ids(sentences)

# Get ELMo embeddings
embeddings = elmo(character_ids)

**Added for this repository:** the article's code prints nothing, so this shows the result. Note that `Elmo` applies dropout (0.5 by default) and a new module starts in training mode, so the embeddings above change from run to run. Calling `elmo.eval()` switches dropout off, so every run of the notebook gives the same vectors. (ELMo's LSTM also carries its state from one call to the next, so a second call on the same sentences, like the one below, gives slightly different numbers: with `elmo.eval()` set before the first call, the similarity below is 0.951; here, on the second call, it is 0.953.)

In [3]:
import torch
elmo.eval()
with torch.no_grad():
    vectors = elmo(character_ids)['elmo_representations'][0]
print('shape (sentences, words, dimensions):', tuple(vectors.shape))
cos = torch.nn.functional.cosine_similarity
print(f"'green' in 'green apple' vs 'green thumb': {cos(vectors[0, 3], vectors[1, 3], dim=0).item():.3f}")
print("A static embedding (Word2Vec, GloVe) would give 1.000 here: ELMo gives 'green' a different vector in each sentence.")

shape (sentences, words, dimensions): (2, 5, 256)
'green' in 'green apple' vs 'green thumb': 0.953
A static embedding (Word2Vec, GloVe) would give 1.000 here: ELMo gives 'green' a different vector in each sentence.
